In [9]:
import pandas as pd
import plotly.express as px
import numpy as np

# Load Global Energy Mix dataset
df = pd.read_csv('/kaggle/input/datasets/josephsudheer0506/global-energy-mix/global_energy_mix.csv')

# Source type mapping
source_category = {
    'Coal': 'Fossil',
    'Oil': 'Fossil',
    'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon',
    'Hydro': 'Low-carbon',
    'Wind': 'Renewable',
    'Solar': 'Renewable',
    'Other Renewables': 'Renewable'
}

df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
display(df.head(10))

Loaded: 103 rows


,Country,Region,Source,Share_pct,TWh,Source_Type
0,United States,North America,Coal,10,1015.0,Fossil
1,United States,North America,Oil,35,3220.0,Fossil
2,United States,North America,Natural Gas,34,3083.0,Fossil
3,United States,North America,Nuclear,9,798.0,Low-carbon
4,United States,North America,Hydro,3,339.0,Low-carbon
5,United States,North America,Wind,4,413.0,Renewable
6,United States,North America,Solar,3,325.0,Renewable
7,United States,North America,Other Renewables,2,229.0,Renewable
8,China,Asia,Coal,58,6929.1,Fossil
9,China,Asia,Oil,18,1620.0,Fossil


In [13]:
import pandas as pd
import plotly.express as px
import numpy as np

# CVD-friendly colours
source_colors = {
    'Coal': '#0072B2',          # Blue
    'Oil': '#E69F00',           # Orange
    'Natural Gas': '#009E73'    # Green
}

# Parent node colour for treemap
parent_color = '#D3D3D3'

print("Dataset ready:", df.shape)
print(df['Source_Type'].value_counts())

Dataset ready: (103, 6)
Source_Type
Fossil        43
Renewable     34
Low-carbon    26
Name: count, dtype: int64


In [18]:
import plotly.graph_objects as go

# Filter fossil fuels
df_fossil = df[df['Source_Type'] == 'Fossil'].copy()

# Aggregate energy consumption
fossil_data = (
    df_fossil.groupby(['Region', 'Country', 'Source'], as_index=False)['TWh']
    .sum()
)

# Create hierarchy lists
labels = ['Fossil Fuels']
parents = ['']
values = [fossil_data['TWh'].sum()]
colors = ['#D3D3D3']

# Add regions
for region in fossil_data['Region'].unique():
    region_data = fossil_data[fossil_data['Region'] == region]

    labels.append(region)
    parents.append('Fossil Fuels')
    values.append(region_data['TWh'].sum())
    colors.append('#BDBDBD')

    # Add countries
    for country in region_data['Country'].unique():
        country_data = region_data[region_data['Country'] == country]

        labels.append(country)
        parents.append(region)
        values.append(country_data['TWh'].sum())
        colors.append('#D3D3D3')

        # Add energy sources
        for _, row in country_data.iterrows():
            labels.append(row['Source'])
            parents.append(country)
            values.append(row['TWh'])
            colors.append(source_colors[row['Source']])

# Find the region with the highest fossil consumption
region_totals = fossil_data.groupby('Region')['TWh'].sum()
top_region = region_totals.idxmax()
top_value = region_totals.max()

# Build treemap
fig1 = go.Figure(go.Treemap(
    labels=labels,
    parents=parents,
    values=values,
    branchvalues='total',
    marker=dict(
        colors=colors,
        line=dict(width=1, color='white')
    ),
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    hovertemplate='<b>%{label}</b><br>%{value:,.1f} TWh<extra></extra>',
    root=dict(color='#D3D3D3')
))

fig1.update_layout(
    title=f'{top_region} has the highest fossil fuel consumption ({top_value:,.1f} TWh)',
    title_x=0.02,
    template='plotly_white',
    margin=dict(t=80, l=10, r=10, b=10)
)

fig1.show()

In [15]:
# Aggregate TWh by Region and Source
region_fossil = (
    df_fossil.groupby(['Region', 'Source'], as_index=False)['TWh']
    .sum()
)

# Calculate each region's fossil energy total
region_fossil['Region_Total'] = (
    region_fossil.groupby('Region')['TWh'].transform('sum')
)

# Convert TWh into percentages
region_fossil['Share_pct'] = (
    region_fossil['TWh'] / region_fossil['Region_Total'] * 100
)

# Find region with highest coal share
coal_shares = (
    region_fossil[region_fossil['Source'] == 'Coal']
    .sort_values('Share_pct', ascending=False)
)

top_coal_region = coal_shares.iloc[0]['Region']
top_coal_share = coal_shares.iloc[0]['Share_pct']

# Sort regions by coal share
region_order = coal_shares['Region'].tolist()

print("Most coal-dependent region:", top_coal_region)
print(f"Coal share: {top_coal_share:.1f}%")

Most coal-dependent region: Africa
Coal share: 78.0%


In [16]:
fig4 = px.bar(
    region_fossil,
    x='Share_pct',
    y='Region',
    color='Source',
    orientation='h',
    barmode='stack',
    category_orders={
        'Region': region_order,
        'Source': ['Coal', 'Oil', 'Natural Gas']
    },
    color_discrete_map={
        'Coal': source_colors['Coal'],
        'Oil': '#A6A6A6',
        'Natural Gas': '#D9D9D9'
    },
    title=f'{top_coal_region} relies most on coal ({top_coal_share:.1f}% of fossil energy)',
    labels={
        'Share_pct': 'Share of regional fossil energy (%)',
        'Region': 'Region',
        'Source': 'Fossil fuel'
    }
)

# Show percentage labels on Coal only
fig4.update_traces(
    texttemplate='%{x:.1f}%',
    textposition='inside',
    selector=dict(name='Coal')
)

fig4.update_layout(
    template='plotly_white',
    title_x=0.02,
    xaxis=dict(range=[0, 100], ticksuffix='%'),
    yaxis={'categoryorder': 'array', 'categoryarray': region_order},
    legend_title='Energy source'
)

fig4.show()

The Task 1 treemap answers which countries and regions contribute the most to total fossil fuel consumption and shows the breakdown by Coal, Oil and Natural Gas.

The 100% stacked bar chart answers which region relies most on coal as a percentage of its total fossil fuel mix. It makes it easier to compare the proportions of Coal, Oil and Natural Gas across regions.

For example, our chart shows that Africa has the highest coal share at 78.0%.